# Learning curves

- `modelling_df4_dimless_p3-NN_2.3-opt_MLP_simple_search.ipynb`: две ячейки после "Save same models, but on CPU" [по одной на каждый таргет]
- `modelling_df4_dimless_p3-NN_3.2-opt_TabNet_splashing_narrow_search.ipynb`: поледняя ячейка
- `modelling_df4_dimless_p3-NN_3.3-opt_TabNet_no_fragmentation_narrow_search.ipynb`: последняя ячейка

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import logging

# Set logging level for PyTorch Tabular
logging.getLogger("pytorch_tabular").setLevel(logging.ERROR)

# Set logging level for PyTorch Lightning
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

# Set logging level for Lightning Fabric
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)

# Disable device availability messages
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.FATAL)

import pandas as pd
import numpy as np

from pytorch_tabular.models import CategoryEmbeddingModelConfig, TabNetModelConfig

import optuna

from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    PytorchTabularEstimator,
    deep_update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

# from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.loggers import TensorBoardLogger

seed = RANDOM_STATE

import os
import shutil
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# Fix error with save weights

# import torch
# # from omegaconf import DictConfig
# # from omegaconf.base import ContainerMetadata
# # import typing
# # torch.serialization.add_safe_globals([DictConfig, ContainerMetadata, typing.Any])
# import pytorch_tabular.utils.python_utils as pt_utils
# pt_utils.pl_load = lambda f, map_location: torch.load(f, map_location=map_location, weights_only=False)

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore")

In [4]:
def process_logs(
        target, postfix, 
        path_results=Path('../results/nn_logs'),
        path_logs=Path('lightning_logs'), 
        ):
    os.makedirs(path_results, exist_ok=True)
    folders_logs=os.listdir('lightning_logs')
    dfs = []
    for folder in folders_logs:
        path_events_file = path_logs / folder / [
            x for x in os.listdir(path_logs / folder) if 'tfevents' in x][0]
        
        event_acc = EventAccumulator(str(path_events_file))
        event_acc.Reload()

        train_losses_events = event_acc.Scalars('train_loss_0')
        valid_losses_events = event_acc.Scalars('valid_loss_0')

        data = []
        for i in range(len(train_losses_events)):
            data.append({
                'step': train_losses_events[i].step,
                'train_loss': train_losses_events[i].value,
                'valid_loss': valid_losses_events[i].value})

        df = pd.DataFrame(data)
        df['folder'] = folder
        dfs.append(df)
    shutil.rmtree(path_logs)
    result_df = pd.concat(dfs)
    result_df.to_excel(path_results / f'{target}_{postfix}.xlsx', index=False)

In [5]:
if os.path.exists('lightning_logs'): shutil.rmtree('lightning_logs')

# Settings

In [6]:
save_model_and_metrics = False
model_postfix = 'opt-model'
metrics_file = ''

## MLP

### Splashing

In [7]:
estimator = PytorchTabularEstimator
target = "splashing"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'dropout': 0.04379983378244369,
        'learning_rate': 0.009487772841911948,
        'layers': '128',
        'activation': 'LeakyReLU',
        'batch_norm_continuous_input': True, # We already have normalized features    
        'use_batch_norm': False, # For hidden layers
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 300, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[CV] END .................................................... total time=   6.4s
[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.3s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.7s
[CV] END .................................................... total time=   0.5s
no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,CategoryEmbeddingModel_splashing_smote_opt-model
holdout_test_f1_macro,0.806892
holdout_test_accuracy_balanced,0.799769
holdout_test_roc_auc,0.918981
holdout_test_f1,0.868687
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.839394
cv_test_accuracy_balanced_median,0.847523
cv_test_roc_auc_median,0.931889


In [8]:
process_logs(target, 'mlp')

### No fragmentation

In [7]:
estimator = PytorchTabularEstimator
target = "no_fragmentation"

add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'dropout': 0.4124773666300698,
        'learning_rate': 0.012011447873654357,
        'layers': '128-64',
        'activation': 'LeakyReLU',
        'batch_norm_continuous_input': True, # We already have normalized features    
        'use_batch_norm': False, # For hidden layers
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 300, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[CV] END .................................................... total time=   6.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   1.1s
[CV] END .................................................... total time=   0.3s
[CV] END .................................................... total time=   0.3s
no summary in estimator class "PytorchTabularEstimator"


,0
target,no_fragmentation
model,CategoryEmbeddingModel_no_fragmentation_smoten...
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.951818
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.90293
cv_test_accuracy_balanced_median,0.915385
cv_test_roc_auc_median,0.97619


In [10]:
process_logs(target, 'mlp')

## TabNet

### Splashing

In [11]:
estimator = PytorchTabularEstimator
target = "splashing"
model_postfix = 'opt-model'
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": TabNetModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'n_d': 12,
        'n_a': 4,
        'n_steps': 2,
        'gamma': 1.4781750460004004,
        'n_shared': 1,
        'n_independent': 2,
        'learning_rate': 0.003182186027654311,
        'virtual_batch_size': 16,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 64, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[CV] END .................................................... total time=   5.4s
[CV] END .................................................... total time=  19.3s
[CV] END .................................................... total time=   6.5s
[CV] END .................................................... total time=  10.7s
[CV] END .................................................... total time=   6.3s
[CV] END .................................................... total time=   7.0s
[CV] END .................................................... total time=   7.4s
no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,TabNetModel_splashing_smote_opt-model
holdout_test_f1_macro,0.813397
holdout_test_accuracy_balanced,0.815972
holdout_test_roc_auc,0.91358
holdout_test_f1,0.863158
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.860788
cv_test_accuracy_balanced_median,0.873839
cv_test_roc_auc_median,0.907121


In [12]:
process_logs(target, 'tabnet')

### No fragmentation

In [13]:
estimator = PytorchTabularEstimator
target = "no_fragmentation"
model_postfix = 'opt-model'
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

estimator_params = {
    "model_class": TabNetModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'n_d': 4,
        'n_a': 8,
        'n_steps': 2,
        'gamma': 1.722004270893931,
        'n_shared': 1,
        'n_independent': 2,
        'learning_rate': 0.00781783794374976,
        'virtual_batch_size': 16,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 64, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[CV] END .................................................... total time=   5.2s
[CV] END .................................................... total time=   7.7s
[CV] END .................................................... total time=   6.4s
[CV] END .................................................... total time=   8.6s
[CV] END .................................................... total time=   5.7s
[CV] END .................................................... total time=   4.6s
[CV] END .................................................... total time=   7.0s
no summary in estimator class "PytorchTabularEstimator"


,0
target,no_fragmentation
model,TabNetModel_no_fragmentation_smotenc_opt-model
holdout_test_f1_macro,0.860566
holdout_test_accuracy_balanced,0.902273
holdout_test_roc_auc,0.923636
holdout_test_f1,0.808511
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.84043
cv_test_accuracy_balanced_median,0.864469
cv_test_roc_auc_median,0.928571


In [14]:
process_logs(target, 'tabnet')